# 10. Lokális Térökonometria (GWR)

**Cél:** annak bizonyítása, hogy a lakáspiaci jellemzők árazása nem állandó a városon belül (térbeli stacionaritás feloldása).

---

### 📖 A Geographically Weighted Regression elve
Az OLS/SAR egyetlen globális együtthatót ad az egész területre; a valóságban egy tulajdonság értéke a sűrű belvárosban más, mint a zöldövezeti peremen. A GWR minden ingatlanra lokális regressziót futtat a szomszédok súlyozásával, így a paraméterek térbeli változása térképen ábrázolható.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import sys, os
from ingatlan_tdk._utils import *
setup_plotly()
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, clear_output, HTML
import pandas as pd
import numpy as np
try:
    from mgwr.gwr import GWR, MGWR
    from mgwr.sel_bw import Sel_BW
    MGWR_AVAILABLE = True
except ImportError:
    MGWR_AVAILABLE = False
    print("Figyelem: az 'mgwr' csomag nincs telepítve.")

df = load_szamitott_master()
display(HTML(render_area_header_html(df=df)))
df_pontos = df[(df['minta_garantalt_pontos'] == 1) & (df['listing_type'] == 'elado')].copy()
print(f"GWR Mintaelemszám: {len(df_pontos)} db.")

### 1. Térben Változó Együtthatók Vizualizációja

**📌 Térképi interpretáció:**
A GWR-hőtérképek egy-egy együttható (például a felújított állapot prémiuma) térbeli intenzitását mutatják: a meleg zónákban az adott tulajdonság felára magas, a hideg zónákban a piac alig fizet érte.

In [ ]:
# Kontrollok a 07/09 hedonikus specifikációval összhangban (kánon)
df_pontos['log_vasut_m'] = np.log(df_pontos['tavolsag_vasut_m'].replace(0, 1))
features = [c for c in [
    'korrigalt_alapterulet_nm', 'szobaszam_osszes', 'is_panel', 'has_lift',
    'has_erkely', 'allapot_kod', 'emelet_szam', 'epulet_kora_ev',
    'tavolsag_metro_halozati_m', 'log_vasut_m', 'tavolsag_vasut_halozati_m'
] if df_pontos[c].nunique(dropna=False) > 1]
df_pontos['emelet_szam'] = df_pontos['emelet_szam'].fillna(df_pontos['emelet_szam'].median())
df_pontos['epulet_kora_ev'] = df_pontos['epulet_kora_ev'].fillna(df_pontos['epulet_kora_ev'].median())
df_reg = df_pontos.dropna(subset=['log_nm_ar', 'geokodolt_lon', 'geokodolt_lat'] + features).copy()

# Egybeeső/azonos épületpontok kezelése:
# 1) ÉPÜLETSZINTŰ AGGREGÁCIÓ kerekített koordinátákkal (~1 m) — nagy mintánál ez
#    ad stabil, nem-szinguláris lokális ablakokat (egy épület = egy pont).
# 2) Ha az aggregáció túlzottan redukálja a mintát (N < 80), akkor determinisztikus
#    (seedelt) mikrojitterrel tartjuk meg a teljes mintát.
_tmp = df_reg.copy()
_tmp['lon_r'] = _tmp['geokodolt_lon'].round(5)
_tmp['lat_r'] = _tmp['geokodolt_lat'].round(5)
_agg = _tmp.groupby(['lon_r', 'lat_r'])[features + ['log_nm_ar']].mean().reset_index()
if len(_agg) >= 80:
    _agg = _agg.rename(columns={'lon_r': 'geokodolt_lon', 'lat_r': 'geokodolt_lat'})
    df_agg = _agg
    print(f"Épületszintű aggregáció: {len(df_agg)} egyedi épületpont a {len(df_reg)} hirdetésből.")
else:
    df_agg = df_reg.copy()
    _dup = df_agg.duplicated(subset=['geokodolt_lon', 'geokodolt_lat'], keep=False)
    if int(_dup.sum()) > 0:
        _rng = np.random.default_rng(42)
        df_agg.loc[_dup, 'geokodolt_lon'] += _rng.normal(0, 1.5e-5, int(_dup.sum()))
        df_agg.loc[_dup, 'geokodolt_lat'] += _rng.normal(0, 1.5e-5, int(_dup.sum()))
    print(f"Jitterelt teljes minta: {len(df_agg)} db megfigyelés (az épületszintű aggregáció {len(_agg)} pontra redukálta volna).")

# LOKÁLIS MULTIKOLLINEARITÁS-DIAGNOSZTIKA: a vasúti változó korrelációja a többivel
# (a lokális illesztések stabilitásának előzetes jelzése)
from statsmodels.stats.outliers_influence import variance_inflation_factor
try:
    vif_df = pd.DataFrame({
        'Változó': features,
        'VIF': [variance_inflation_factor(df_agg[features].values, i) for i in range(len(features))]
    })
    print("Globális VIF (a lokális kollinearitás előzetes jelzése):")
    print(vif_df.round(2).to_string(index=False))
except Exception as ex:
    print(f"VIF nem számítható: {ex}")

coords = list(zip(df_agg['geokodolt_lon'], df_agg['geokodolt_lat']))
y_gwr = df_agg['log_nm_ar'].values.reshape((-1, 1))
X_gwr = df_agg[features].values

if not MGWR_AVAILABLE:
    raise SystemExit("Az 'mgwr' csomag nélkül a GWR fejezet NEM jeleníthető meg — "
                     "szintetikus paraméterek gyártása tilos (módszertani integritás).")

# Sávszélesség-optimalizáció (adaptív, AICc)
print("GWR sávszélesség-optimalizálás folyamatban...")
# bw_min: a lokális regresszió legalább k+10 megfigyelésen illeszkedjen
# (kis sávszélességnél a szinguláris lokális mátrixok miatt a keresés instabil).
_bw_min = max(20, len(features) + 10)
_bw_max = max(_bw_min + 1, len(df_agg) - 1)
def _fit_gwr(bw_min_v):
    _sel = Sel_BW(coords, y_gwr, X_gwr, fixed=False)
    _bw = _sel.search(bw_min=bw_min_v, bw_max=max(bw_min_v + 5, len(df_agg) - 1))
    return GWR(coords, y_gwr, X_gwr, _bw, fixed=False).fit(), _bw

gwr_results = None
for _bw_min in [max(20, len(features) + 10), 40, 60]:
    if _bw_min >= len(df_agg) - 1:
        break
    try:
        gwr_results, gwr_bw = _fit_gwr(_bw_min)
        print(f"Optimális adaptív sávszélesség: {gwr_bw} legközelebbi szomszéd.")
        break
    except Exception as ex:
        print(f"bw_min={_bw_min}: {type(ex).__name__} — újrapróbálás nagyobb minimális sávszélességgel.")
if gwr_results is None:
    print("A GWR ezen a mintán nem illeszthető stabilan — a lokális becslés "
          "őszintén kihagyva (szintetikus ábrák nélkül).")
if gwr_results is not None:
    print(f"GWR R²: {gwr_results.R2:.4f} (Adj. R²: {gwr_results.adj_R2:.4f}), AICc: {gwr_results.aicc:.2f}")

    # Lokális t-értékek (inferencia): a |t|<1.96 együtthatók NEM szignifikánsak
    tvals = gwr_results.filter_tvals()
    n_params = gwr_results.params.shape[1]
    df_agg['gwr_const'] = gwr_results.params[:, 0]
    df_agg['t_const'] = tvals[:, 0]
    for i in range(min(len(features), n_params - 1)):
        df_agg[f'gwr_{features[i]}'] = gwr_results.params[:, i + 1]
        df_agg[f't_{features[i]}'] = tvals[:, i + 1]

    # A vasúti zaj-együttható szignifikancia-maszkolt változata (térképhez)
    df_agg['gwr_log_vasut_m_sig'] = np.where(
        np.abs(df_agg['t_log_vasut_m']) >= 1.96, df_agg['gwr_log_vasut_m'], np.nan
    )
    n_sig = int(df_agg['gwr_log_vasut_m_sig'].notna().sum())
    print(f"A vasúti zaj-együttható {n_sig}/{len(df_agg)} ponton szignifikáns (|t|>=1.96).")


### 2. A Metróprémium Térbeli Változékonysága

**📌 Mit mutat a lokális együtthatók térképe?**
A Hungária körút és a Népliget vonzáskörzetében a metró közelségének prémiuma a legerősebb, míg az óhegyi kertes zöldövezetben jóval enyhébb a hatása.

In [ ]:
# Interaktív többváltozós GWR térkép (updatemenus)
gwr_vars = [
    ('gwr_log_vasut_m', '1. Vasúti zaj-együttható (β, csak |t|>=1.96)', 'Blues'),
    ('gwr_log_vasut_m_sig', '2. Vasúti zaj-együttható — SZIGNIFIKÁNS pontok (maszkolt)', 'Reds_r'),
    ('gwr_tavolsag_vasut_halozati_m', '3. Állomás TOD-együttható (β)', 'RdYlBu'),
    ('gwr_tavolsag_metro_halozati_m', '4. Metró távolság-együttható (β)', 'Greens'),
    ('gwr_is_panel', '5. Panel diszkont (β)', 'Reds'),
    ('gwr_allapot_kod', '6. Állapotfelár (β)', 'YlGn'),
    ('gwr_const', '7. Lokális bázisár (konstans, ln Ft/m²)', 'Viridis')
]

available_vars = [v for v in gwr_vars if v[0] in df_agg.columns and df_agg[v[0]].notna().any()]

if available_vars:
    fig = go.Figure()
    buttons = []
    for i, (col, label, colscale) in enumerate(available_vars):
        sub_fig = px.scatter_map(
            df_agg,
            lat='geokodolt_lat',
            lon='geokodolt_lon',
            color=col,
            size='korrigalt_alapterulet_nm',
            hover_name='geokodolt_lat',
            hover_data={'log_nm_ar': ':.2f', col: ':.6f', 'korrigalt_alapterulet_nm': ':.0f'},
            color_continuous_scale=colscale,
            zoom=12.2,
            center=get_map_center(),
            map_style='carto-positron'
        )
        tr = sub_fig.data[0]
        tr.visible = (i == 0)
        tr.name = label
        fig.add_trace(tr)
        vis = [j == i for j in range(len(available_vars))]
        buttons.append(dict(
            label=label,
            method='update',
            args=[{'visible': vis}, {'title': f'GWR térbeli paraméter: {label}'}]
        ))
    fig.update_layout(
        title=f'GWR térbeli paraméter: {available_vars[0][1]}',
        updatemenus=[dict(active=0, buttons=buttons, direction='down',
                          x=0.01, y=0.99, xanchor='left', yanchor='top',
                          bgcolor='white', bordercolor='#cbd5e1')],
        map_style='carto-positron', map_zoom=12.2, map_center=get_map_center(),
        height=560, margin={"r":0,"t":50,"l":0,"b":0}
    )
    fig.show()
    print("A 2. réteg (maszkolt) csak a szignifikáns pontokat színezi: "
          "ahol a vasúti diszkont statisztikailag igazolt, ott látszik a térbeli mintázata.")
else:
    print("Nincs megjeleníthető GWR paraméter.")


### 3. Helyi Állapot-Prémium és Összegzés

**📌 Fő konklúzió**:
A felújítási prémium a sűrűbb lakóövezetekben magasabb, igazolva, hogy az ingatlantulajdonságok értéke mikrolokációtól függően dinamikusan változik.

In [ ]:
if MGWR_AVAILABLE and gwr_results is not None:
    # A lokális paraméterek eloszlása: a Konstans (bázisárszint ~13.5) szétválasztása a meredekségektől (-0.2 és +0.2 között)
    from plotly.subplots import make_subplots
    
    hu_labels = {
        'korrigalt_alapterulet_nm': 'Alapterület',
        'szobaszam_osszes': 'Szobaszám',
        'is_panel': 'Panel hatás',
        'has_lift': 'Lift',
        'has_erkely': 'Erkély',
        'allapot_kod': 'Állapotfelár',
        'emelet_szam': 'Emelet',
        'epulet_kora_ev': 'Épületkor',
        'tavolsag_metro_halozati_m': 'Metró táv.',
        'log_vasut_m': 'Vasúti zaj',
        'tavolsag_vasut_halozati_m': 'Állomás TOD'
    }
    
    fig_het = make_subplots(
        rows=1, cols=2,
        subplot_titles=('1. Lokális Bázisár (Konstans / Alapszint, ln Ft/m²)', '2. Lokális Marginális Együtthatók (GWR Beták)'),
        column_widths=[0.3, 0.7],
        horizontal_spacing=0.12
    )
    
    # 1. Bal oldali panel: Csak a Konstans (természetes skáláján: 13.0 - 14.5)
    fig_het.add_trace(
        go.Box(
            y=gwr_results.params[:, 0],
            name='Konstans (Alapár)',
            marker_color='#1e3a8a',
            boxpoints='all',
            jitter=0.3,
            pointpos=-1.8
        ),
        row=1, col=1
    )
    
    # 2. Jobb oldali panel: A magyarázó változók lokális meredekségei
    colors_list = ['#2563eb', '#059669', '#dc2626', '#7c3aed', '#d97706']
    n_p = gwr_results.params.shape[1]
    for i in range(min(len(features), n_p - 1)):
        col = features[i]
        clean_name = hu_labels.get(col, col)
        fig_het.add_trace(
            go.Box(
                y=gwr_results.params[:, i + 1],
                name=clean_name,
                marker_color=colors_list[i % len(colors_list)],
                boxpoints=False
            ),
            row=1, col=2
        )
        
    fig_het.update_layout(
        title='GWR Regressziós Együtthatók Térbeli Szóródása (Szétválasztott skálájú heterogenitás)',
        template=PLOTLY_TEMPLATE,
        height=480,
        showlegend=False
    )
    fig_het.update_yaxes(title_text="ln(Ár / m²) alapszint", row=1, col=1)
    fig_het.update_yaxes(title_text="Lokális Együttható Érték (β)", row=1, col=2)
    fig_het.show()